# 🎬 Cinematic Engine V17 — CLEAN v9.0
### شغّل الـ cells بالترتيب: 1 ثم 2 ثم 3
**قبل أي حاجة:** `Runtime → Change runtime type → T4 GPU → Save`

> ⚠️ لو عملت Restart Runtime، لازم تشغّل CELL 1 و CELL 2 من أول


In [ ]:
# ════════════════════════════════════════
# CELL 1 — SETUP
# شغّله مرة واحدة في بداية كل session
# ════════════════════════════════════════

import os
import sys
import shlex
import subprocess
import torch
import nltk

# ─── Constants ───────────────────────────────────────────────────────────────
# NOTE: PROJECT_DIR is intentionally re-declared in each cell so every cell
# can run independently after a Kernel restart without relying on prior state.
REPO_URL    = 'https://github.com/michelluke1984-cyber/cinematic-engine'
PROJECT_DIR = '/content/cinematic-engine'

# (filename, download_url) — weights are downloaded into PROJECT_DIR
MODEL_URLS = [
    (
        'GFPGANv1.3.pth',
        'https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.3.pth',
    ),
    (
        'realesr-general-x4v3.pth',
        'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-general-x4v3.pth',
    ),
]

# (pip_package_spec, human_label)
# Packages are ordered so that critical dependencies come first.
# Failed packages are logged but do NOT stop the loop — partial installs
# are preferable to a full abort during environment setup.
_CRITICAL_PACKAGES = [
    ('diffusers==0.27.2 transformers==4.40.0 accelerate==0.30.0', 'diffusers'),
    ('xformers==0.0.26.post1 einops',                             'xformers'),
]
_OPTIONAL_PACKAGES = [
    ('gfpgan realesrgan nltk sentencepiece tqdm',  'GFPGAN'),
    ('"gradio>=4.0.0,<5.0.0" bitsandbytes Pillow', 'Gradio'),
    ('insightface==0.7.3 onnxruntime-gpu',          'InsightFace'),
    ('"websockets>=10.4,<13.0" nest_asyncio',       'WebSocket'),
]
PACKAGES = _CRITICAL_PACKAGES + _OPTIONAL_PACKAGES


# ─── Helpers ─────────────────────────────────────────────────────────────────
def run_cmd(args: list[str], label: str = '') -> subprocess.CompletedProcess:
    """
    Run a command from an argument list (shell=False for safety).
    Raises RuntimeError on non-zero exit, surfacing a sanitised stderr excerpt.

    Args are truncated to the first 6 tokens before being included in the
    error message to avoid accidentally leaking tokens or credentials that
    may be present in environment-injected arguments.
    stderr is suppressed on success; many tools (pip, git, wget) write
    informational output to stderr even on successful runs.
    """
    result = subprocess.run(args, capture_output=True, text=True)
    if result.returncode != 0:
        prefix   = f'[{label}] ' if label else ''
        safe_cmd = ' '.join(str(a) for a in args[:6])
        raise RuntimeError(
            f'❌ {prefix}Command failed: {safe_cmd}\n'
            f'{result.stderr[:400]}'
        )
    return result


def ensure_project_dir() -> None:
    """Clone the repo if missing; attempt a quiet pull if already present."""
    if not os.path.exists(PROJECT_DIR):
        print('⬇️  تحميل المشروع...')
        run_cmd(['git', 'clone', REPO_URL, PROJECT_DIR], 'git clone')
        print('✅ المشروع جاهز')
    else:
        result = subprocess.run(
            ['git', '-C', PROJECT_DIR, 'pull', '-q'],
            capture_output=True, text=True,
        )
        if result.returncode != 0:
            print(f'⚠️  git pull فشل — سيكمل بالنسخة الموجودة:\n{result.stderr[:120]}')
        else:
            print('✅ المشروع محدّث')


def install_packages() -> None:
    """
    Install all pip packages listed in PACKAGES.
    Each group is installed in a single pip invocation.
    Failures are logged; installation continues for the remaining groups.
    """
    print('\n📦 تثبيت الـ packages...')
    for spec, label in PACKAGES:
        print(f'  {label}...', end=' ', flush=True)
        args = [sys.executable, '-m', 'pip', 'install', '-q'] + shlex.split(spec)
        try:
            run_cmd(args, label)
            print('✅')
        except RuntimeError as exc:
            print(f'❌\n  {exc}')


def download_model_weights(project_dir: str) -> None:
    """
    Download model weight files into *project_dir* using wget.
    Skips files that already exist on disk.
    """
    print('\n⬇️  تحميل الـ models...')
    for filename, url in MODEL_URLS:
        dest = os.path.join(project_dir, filename)
        if os.path.exists(dest):
            print(f'  ✅ {filename} موجود')
            continue
        run_cmd(['wget', '-q', '-O', dest, url], filename)
        print(f'  ✅ {filename}')


# ─── Main setup flow ─────────────────────────────────────────────────────────
run_cmd([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'], 'pip upgrade')

if not torch.cuda.is_available():
    raise RuntimeError('❌ مفيش GPU — Runtime → Change runtime type → T4 GPU')
gpu_props = torch.cuda.get_device_properties(0)
print(f'✅ GPU: {torch.cuda.get_device_name(0)} | VRAM: {gpu_props.total_memory / 1e9:.1f} GB')

ensure_project_dir()
os.chdir(PROJECT_DIR)

install_packages()

nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)

download_model_weights(PROJECT_DIR)

print('\n✅ CELL 1 خلص — شغّل CELL 2')


In [ ]:
# ════════════════════════════════════════
# CELL 2 — تحميل الـ Engine
# ════════════════════════════════════════

import __main__ as _main
import importlib.util
import os
import sys
import torch

# ─── Constants ───────────────────────────────────────────────────────────────
# NOTE: PROJECT_DIR is intentionally re-declared in each cell so every cell
# can run independently after a Kernel restart without relying on prior state.
PROJECT_DIR            = '/content/cinematic-engine'
ENGINE_FILE            = os.path.join(PROJECT_DIR, 'cinematic_engine_v17_pro.py')
VRAM_WARN_THRESHOLD_GB = 10.0


# ─── Helpers ─────────────────────────────────────────────────────────────────
def _load_module_from_file(name: str, filepath: str):
    """
    Load a Python file as a named module and register it in sys.modules.

    Invalidates any previously cached version of *name* before loading so
    re-running the cell always picks up the latest file on disk.
    Rolls back the sys.modules entry on any import-time exception so the
    module name is never left in a partially-initialised state.
    """
    sys.modules.pop(name, None)  # invalidate stale cache
    spec   = importlib.util.spec_from_file_location(name, filepath)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    try:
        spec.loader.exec_module(module)
    except Exception as exc:
        sys.modules.pop(name, None)  # rollback on failure
        raise RuntimeError(
            f'❌ فشل تحميل {os.path.basename(filepath)}\n'
            f'   السبب: {type(exc).__name__}: {exc}\n'
            f'   تأكد إن الملف موجود وفيه syntax صح'
        ) from exc
    return module


def _purge_stale_engine_symbols() -> None:
    """
    Remove the previously loaded engine module and its exported symbols from
    sys.modules and the __main__ namespace so re-running this cell is safe.

    Tracks every public symbol exported from the stale 'cev17' module and
    removes each one by name, guaranteeing a clean slate on every reload
    regardless of what names the engine happens to export.
    """
    stale_module = sys.modules.get('cev17')
    stale_names: set[str] = (
        {k for k in vars(stale_module) if not k.startswith('_')}
        if stale_module is not None
        else set()
    )

    sys.modules.pop('cev17', None)

    # Remove every previously exported symbol from __main__ and globals().
    for key in stale_names:
        vars(_main).pop(key, None)
        globals().pop(key, None)


def _check_vram(threshold_gb: float = VRAM_WARN_THRESHOLD_GB) -> None:
    """
    Report free VRAM and warn when it falls below *threshold_gb*.

    Uses torch.cuda.mem_get_info() for a driver-level query that accounts
    for PyTorch's reserved-but-not-allocated cache, giving an accurate
    reading of physically available VRAM regardless of caching state.
    Callers can pass a custom *threshold_gb* to suit different model sizes.
    """
    free_vram_gb, total_vram_gb = (v / 1e9 for v in torch.cuda.mem_get_info(0))
    print(f'✅ VRAM: {free_vram_gb:.1f} GB free / {total_vram_gb:.1f} GB total')
    if free_vram_gb < threshold_gb:
        print(
            f'⚠️  VRAM حر ({free_vram_gb:.1f} GB) أقل من المثالي '
            f'({threshold_gb} GB) — قد يحدث OOM مع FLUX DEV'
        )


# ─── Guards ──────────────────────────────────────────────────────────────────
if not os.path.exists(PROJECT_DIR):
    raise RuntimeError('❌ شغّل CELL 1 الأول')
os.chdir(PROJECT_DIR)

# Ensure PROJECT_DIR is on sys.path so relative imports inside the engine work.
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

if not os.path.exists(ENGINE_FILE):
    raise RuntimeError(f'❌ الملف مش موجود: {ENGINE_FILE}')

# ─── Load engine ─────────────────────────────────────────────────────────────
print('⏳ تحميل CinematicEngineV17...')
_purge_stale_engine_symbols()

cev17 = _load_module_from_file('cev17', ENGINE_FILE)

# Re-export all public symbols into the notebook's global namespace so that
# CinematicEngineV17 and its companions are directly accessible in Cell 3.
for _name, _val in vars(cev17).items():
    if not _name.startswith('_'):
        globals()[_name] = _val

if 'CinematicEngineV17' not in globals():
    raise RuntimeError('❌ CinematicEngineV17 مش موجودة في الملف')

# ─── Instantiate engine ──────────────────────────────────────────────────────
print('⏳ Initialising engine...')
_check_vram()

try:
    engine = CinematicEngineV17()  # noqa: F821  (injected into globals above)
except Exception as exc:
    raise RuntimeError(
        f'❌ فشل تشغيل الـ Engine: {type(exc).__name__}: {exc}\n'
        f'   تأكد إن الـ VRAM كافية وإن الـ models اتحملت'
    ) from exc

print(f'✅ Engine: {type(engine).__name__}')
print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
print('\n✅ CELL 2 خلص — شغّل CELL 3')


In [ ]:
# ════════════════════════════════════════
# CELL 3 — تشغيل الـ Bridge
# هيفضل شغّال — اضغط ⬛ عشان توقف
# ════════════════════════════════════════

import asyncio
import importlib.util
import os
import re
import subprocess
import sys
import threading
import time

import nest_asyncio

nest_asyncio.apply()

# ─── Constants ───────────────────────────────────────────────────────────────
# NOTE: PROJECT_DIR is intentionally re-declared in each cell so every cell
# can run independently after a Kernel restart without relying on prior state.
PROJECT_DIR        = '/content/cinematic-engine'
BRIDGE_FILE        = os.path.join(PROJECT_DIR, 'cev17_backend.py')
CLOUDFLARED_BIN    = '/tmp/cloudflared'
CLOUDFLARED_URL    = (
    'https://github.com/cloudflare/cloudflared/releases/latest/download/'
    'cloudflared-linux-amd64'
)
SERVER_PORT        = 7860
TUNNEL_TIMEOUT_SEC = 45

# Matches only genuine tunnel URLs emitted by cloudflared on connection.
# Anchored to known log prefixes (url= / https) so example URLs that appear
# inside error messages cannot be mistakenly captured as the real address.
_TUNNEL_URL_RE = re.compile(
    r'(?:url=|https).*?(https://[\w-]+\.trycloudflare\.com)'
)


# ─── Module loader ────────────────────────────────────────────────────────────
# The canonical implementation lives in Cell 2. This fallback activates only
# when Cell 3 is run standalone after a Kernel restart (without Cell 2),
# honouring the per-cell independence contract.
if '_load_module_from_file' not in dir():
    def _load_module_from_file(name: str, filepath: str):
        """
        Fallback module loader — identical behaviour to the Cell 2 version.
        See Cell 2 for the full rationale and documentation.
        """
        sys.modules.pop(name, None)
        spec   = importlib.util.spec_from_file_location(name, filepath)
        module = importlib.util.module_from_spec(spec)
        sys.modules[name] = module
        try:
            spec.loader.exec_module(module)
        except Exception as exc:
            sys.modules.pop(name, None)
            raise RuntimeError(
                f'❌ فشل تحميل {os.path.basename(filepath)}\n'
                f'   السبب: {type(exc).__name__}: {exc}\n'
                f'   تأكد إن الملف موجود وفيه syntax صح'
            ) from exc
        return module


# ─── Helpers ─────────────────────────────────────────────────────────────────
def _get_engine():
    """Retrieve the engine instance from the notebook globals; raise if absent."""
    eng = globals().get('engine')
    if eng is None:
        raise RuntimeError(
            '❌ Engine مش موجود\n'
            '   لو عملت Restart Runtime، لازم تشغّل CELL 1 و CELL 2 من أول\n'
            '   شغّل CELL 2 الأول ثم ارجع لهنا'
        )
    return eng


def _ensure_cloudflared() -> None:
    """Download the cloudflared binary once and make it executable."""
    if os.path.exists(CLOUDFLARED_BIN):
        return
    print('⬇️  تحميل cloudflared...')
    result = subprocess.run(
        ['wget', '-q', '-O', CLOUDFLARED_BIN, CLOUDFLARED_URL],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(
            f'❌ فشل تحميل cloudflared:\n{result.stderr[:200]}'
        )
    os.chmod(CLOUDFLARED_BIN, 0o755)
    print('✅ cloudflared جاهز')


def _free_port(port: int) -> None:
    """Kill any process holding *port* and wait briefly for the socket to close."""
    subprocess.run(['pkill', '-f', f':{port}'], capture_output=True)
    time.sleep(1)


class _TunnelManager:
    """
    Starts a Cloudflare tunnel in a background daemon thread and exposes the
    resulting public URL through a thread-safe interface.

    The ``_ready`` event is set as soon as the background thread exits (with
    or without finding a URL), so :meth:`get_url` will never block longer
    than *timeout* seconds regardless of the tunnel outcome.

    The cloudflared ``Popen`` handle is stored unconditionally in the
    ``finally`` block of :meth:`_run` so :meth:`close` can always terminate
    the process — even when the tunnel exits before producing a URL.
    """

    def __init__(self, port: int) -> None:
        self._port  = port
        self._url:  str | None              = None
        self._proc: subprocess.Popen | None = None
        self._lock  = threading.Lock()
        self._ready = threading.Event()

    def start(self) -> None:
        """Launch the tunnel thread (daemon — stops automatically with the process)."""
        threading.Thread(target=self._run, daemon=True).start()

    def get_url(self, timeout: float = TUNNEL_TIMEOUT_SEC) -> str | None:
        """Block until the tunnel URL is known or *timeout* elapses."""
        self._ready.wait(timeout=timeout)
        with self._lock:
            return self._url

    def close(self) -> None:
        """Terminate the cloudflared process if it is still running."""
        with self._lock:
            proc = self._proc
        if proc is None:
            return
        try:
            proc.terminate()
            print('✅ Tunnel closed')
        except OSError as exc:
            print(f'⚠️  Could not terminate tunnel process: {exc}')

    # ── private ──────────────────────────────────────────────────────────────
    def _run(self) -> None:
        """
        Background thread: launch cloudflared and parse its output for the
        public tunnel URL.

        The process handle is stored via the finally block so close() can
        terminate it regardless of whether a URL was successfully found.
        """
        proc = None
        try:
            proc = subprocess.Popen(
                [
                    CLOUDFLARED_BIN, 'tunnel',
                    '--url', f'http://localhost:{self._port}',
                    '--no-autoupdate',
                ],
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
            )
            for line in proc.stdout:
                match = _TUNNEL_URL_RE.search(line)
                if match:
                    with self._lock:
                        self._url = match.group(1)
                    self._ready.set()
                    return  # URL found — cloudflared keeps running in background
        finally:
            # Guarantee proc is always stored and _ready is always set so
            # close() can act and get_url() callers are never left blocking.
            with self._lock:
                if self._proc is None:
                    self._proc = proc
            self._ready.set()


# ─── Server coroutine ─────────────────────────────────────────────────────────
async def _run_server(bridge_module, engine) -> None:
    """Instantiate the bridge server and run it until cancelled."""
    server = bridge_module.CinematicBridgeServer(engine=engine)
    await server.start()


# ─── Guards ──────────────────────────────────────────────────────────────────
if not os.path.exists(PROJECT_DIR):
    raise RuntimeError('❌ شغّل CELL 1 الأول')
os.chdir(PROJECT_DIR)

if not os.path.exists(BRIDGE_FILE):
    raise RuntimeError(f'❌ {BRIDGE_FILE} مش موجود')

# ─── Main bridge flow ────────────────────────────────────────────────────────
_engine = _get_engine()
print(f'✅ Engine: {type(_engine).__name__}')

_bridge_mod = _load_module_from_file('cev17_bridge', BRIDGE_FILE)
if not hasattr(_bridge_mod, 'CinematicBridgeServer'):
    raise RuntimeError('❌ CinematicBridgeServer مش موجودة في cev17_backend.py')
print('✅ Bridge loaded')

_free_port(SERVER_PORT)
_ensure_cloudflared()

print('⏳ جاري فتح الـ tunnel...')
_tunnel = _TunnelManager(port=SERVER_PORT)
_tunnel.start()

public_url = _tunnel.get_url(timeout=TUNNEL_TIMEOUT_SEC)
if not public_url:
    _tunnel.close()
    raise RuntimeError('❌ فشل فتح الـ Cloudflare tunnel — أعد تشغيل CELL 3')

ws_url = public_url.replace('https://', 'wss://')
print('\n' + '=' * 50)
print('🌐 الـ URL بتاعك:')
print(f'   {ws_url}')
print('=' * 50)
print('1. افتح الـ Dashboard في المتصفح')
print('2. اضغط WS: OFF')
print('3. الصق الـ URL واضغط Connect')
print('=' * 50)
print()

# ─── Run async server ────────────────────────────────────────────────────────
# Use get_running_loop() inside Jupyter/Colab where a loop already exists,
# and fall back to creating a new loop only in plain-Python environments.
# asyncio.run() is intentionally avoided — it creates a second loop that
# conflicts with nest_asyncio in the Colab/Jupyter environment.
try:
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
    loop.run_until_complete(_run_server(_bridge_mod, _engine))
except KeyboardInterrupt:
    print('\n⏹ Bridge stopped')
except Exception as exc:
    print(f'\n❌ خطأ في الـ bridge: {type(exc).__name__}: {exc}')
finally:
    _tunnel.close()
